# Queue: Option B (a) HHD-only topline + (b) 50k bigram HHD fine-tune, seeds 42 / 43 / 44 (6 runs)

Data: `cyttic/hhd-optionb-split` (paragraph- and writer-disjoint HHD split). Each run pushes its model + results to its own public HF repo; a run whose repo already holds `results.json` is skipped, so re-running resumes the queue. A failed run's executed notebook is uploaded to the public HF dataset `cyttic/a5000-queue-logs`.

Start on the server: `./run.sh optionb/queue_optionb.ipynb` (inside tmux).


In [ ]:
import os, json, time, traceback
import papermill as pm
from huggingface_hub import HfApi
NBS = [
 "optionb_hhdonly_ctc_s42_colab.ipynb",
 "optionb_hhdft_human10bi_llmgen_50k_s42_colab.ipynb",
 "optionb_hhdonly_ctc_s43_colab.ipynb",
 "optionb_hhdft_human10bi_llmgen_50k_s43_colab.ipynb",
 "optionb_hhdonly_ctc_s44_colab.ipynb",
 "optionb_hhdft_human10bi_llmgen_50k_s44_colab.ipynb"
]
HERE = os.getcwd(); BASE = os.path.expanduser('~/work/queue_optionb'); os.makedirs(BASE, exist_ok=True)
api = HfApi()
LOGS = 'cyttic/a5000-queue-logs'   # PUBLIC dataset: executed notebooks of failed runs (tracebacks)
api.create_repo(LOGS, repo_type='dataset', private=False, exist_ok=True)

def push_log(path, name):
    try: api.upload_file(path_or_fileobj=path, path_in_repo=f'{name}.ipynb', repo_id=LOGS, repo_type='dataset', commit_message=f'failed run {name}'); print('log ->', f'{LOGS}/{name}.ipynb', flush=True)
    except Exception as e: print('log upload failed', repr(e)[:200], flush=True)

def out_repo(nb):
    src = ''.join(''.join(c['source']) for c in json.load(open(os.path.join(HERE, nb)))['cells'])
    return src.split('OUT_REPO = "', 1)[1].split('"', 1)[0]

def done(repo):
    try: return 'results.json' in api.list_repo_files(repo)
    except Exception: return False

status = {}
for i, nb in enumerate(NBS, 1):
    repo = out_repo(nb); name = nb.replace('_colab.ipynb', '')
    if done(repo): print(f'[{i}/{len(NBS)}] {name}: already on HF, skipped', flush=True); status[name] = 'skipped (done)'; continue
    wd = os.path.join(BASE, name); os.makedirs(wd, exist_ok=True); t0 = time.time()
    print(f'[{i}/{len(NBS)}] {name} -> {repo}  started {time.strftime("%H:%M")}', flush=True)
    try:
        pm.execute_notebook(os.path.join(HERE, nb), os.path.join(wd, nb), cwd=wd, log_output=True, kernel_name='python3')
        status[name] = f'ok ({(time.time() - t0) / 3600:.2f} h)' if done(repo) else 'finished, but results.json not on HF'
    except Exception as e:
        status[name] = 'FAILED: ' + repr(e)[:300]; traceback.print_exc(); push_log(os.path.join(wd, nb), 'queue_optionb__' + name)
    print(f'[{i}/{len(NBS)}] {name}: {status[name]}', flush=True)
print('\n==== queue summary'); [print(f'{k:60s} {v}') for k, v in status.items()]

## Results on HF

In [ ]:
from huggingface_hub import hf_hub_download
for nb in NBS:
    repo = out_repo(nb)
    if not done(repo): print(f'{repo}: no results'); continue
    T = json.load(open(hf_hub_download(repo, 'results.json', force_download=True)))['final']['test_decoding']
    print(repo, {k.split(' (')[0]: round(100 * v['cer'], 2) for k, v in T.items()})